# MLP — Dengue Weekly Forecasting by UF

MISO (Multiple Input, Single Output) com Monte Carlo Dropout para intervalos preditivos.
Hiperparâmetros selecionados automaticamente por grid search interno.

### 1. Imports & Configuration

In [1]:
import warnings, time, itertools
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path
from copy import deepcopy

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

from statsmodels.tsa.stattools import kpss
from statsmodels.regression.linear_model import OLS
from statsmodels.stats.diagnostic import het_breuschpagan

warnings.filterwarnings("ignore")
torch.manual_seed(42)
np.random.seed(42)

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
BASE_DIR = Path(r"C:\myenv\env\Mestrado\2026_imdc_challenge")
DB_DIR   = BASE_DIR / "0_databases"
OUT_DIR  = BASE_DIR / "6_results_mlp_v2"
OUT_DIR.mkdir(exist_ok=True)

# ---------------------------------------------------------------------------
# Global constants
# ---------------------------------------------------------------------------
TARGET    = "casos"
FREQ      = 52        # semanas/ano
PI_LEVELS = [0.50, 0.80, 0.90, 0.95]
N_MC      = 500       # forward passes Monte Carlo Dropout
BATCH_SIZE = 32
PATIENCE   = 30       # early stopping

DROP_EXOG = {
    "pop_density_km2","BSh_koppen","As_koppen","rel_humid_med",
    "casos","date","year","epiweek","uf","uf_code",
    "regional_geocode","regional_health_name","macroregion_code","macroregion_name",
    "train_1","train_2","train_3","train_4",
    "target_1","target_2","target_3","target_4",
    "regional_health_area_km2", "precip_med", "rainy_days", "temp_med", 
    "enso"
}

VALIDATIONS = [
    (1, "train_1", "target_1"),
    (2, "train_2", "target_2"),
    (3, "train_3", "target_3"),
    (4, "train_4", "target_4"),
]

# Hyperparameter search space
HP_GRID = {
    "window_size"  : [8, 13, 26],            # 2 meses, quarter, semestre
    "hidden_layers": [[64, 32], [128, 64, 32]],
    "lr"           : [1e-3, 5e-4],
    "dropout_p"    : [0.1, 0.2],
}
EPOCHS_SEARCH = 200   # épocas por candidato no grid search (com early stopping)
EPOCHS_FINAL  = 400   # épocas no fit final do melhor modelo


### 2. Data Loading & UF Aggregation

In [2]:
print("Carregando hierarch_forecast_lagged.parquet...")
hf_raw = pd.read_parquet(DB_DIR / "hierarch_forecast_lagged.parquet")
hf_raw = hf_raw[hf_raw["uf"] != "ES"]
hf_raw["date"]  = pd.to_datetime(hf_raw["date"])
hf_raw["casos"] = hf_raw["casos"].astype(float)
hf_raw["enso_lag26"]  = hf_raw["enso_lag26"].ffill().bfill()
hf_raw["precip_med_lag11"]  = hf_raw["precip_med_lag11"].ffill().bfill()
hf_raw["rainy_days_lag17"]  = hf_raw["rainy_days_lag17"].ffill().bfill()
hf_raw["temp_med_lag15"]  = hf_raw["temp_med_lag15"].ffill().bfill()

koppen_cols = [c for c in hf_raw.columns if c.endswith("_koppen")]
biome_cols  = [c for c in hf_raw.columns if c.endswith("_biome")]
dummy_cols  = koppen_cols + biome_cols

climate_cols = ["pressure_med", "thermal_range", "precip_med_lag11", 
                "rainy_days_lag17", "temp_med_lag15", "enso_lag26"]

flag_cols    = ["train_1","train_2","train_3","train_4",
                "target_1","target_2","target_3","target_4"]

agg_dict = {
    "casos": "sum", 
    "population": "sum",
    "uf_code": "first", 
    "macroregion_code": "first", 
    "macroregion_name": "first",
    "year": "first", 
    "epiweek": "first",
    **{c: "mean" for c in climate_cols},
    **{c: "sum"  for c in dummy_cols},
    **{c: "first" for c in flag_cols},
}

hf = (hf_raw.groupby(["uf","date"], sort=True).agg(agg_dict).reset_index())
hf[dummy_cols] = hf[dummy_cols].clip(upper=1)
for c in flag_cols:
    hf[c] = hf[c].astype(bool)
hf = hf.sort_values(["uf","date"]).reset_index(drop=True)

# Feature columns for MISO (same set as SARIMAX exogenous + target)
exog_cols  = sorted(set(hf.columns) - DROP_EXOG
                    - {c for c in hf.columns if hf[c].dtype == object})
# TARGET goes first so index 0 is always 'casos' in the window matrix
FEATURE_COLS = [TARGET] + [c for c in exog_cols if c != TARGET]
n_features   = len(FEATURE_COLS)
target_idx   = 0   # 'casos' is always at position 0

uf_code_map = hf[["uf","uf_code"]].drop_duplicates("uf").set_index("uf")
ufs = sorted(hf["uf"].unique())

print(f"  UFs : {len(ufs)}")
print(f"  Features MISO ({n_features}): {FEATURE_COLS[:6]} ... + {n_features-6} more")


Carregando hierarch_forecast_lagged.parquet...
  UFs : 26
  Features MISO (21): ['casos', 'Af_koppen', 'Am_koppen', 'Amazônia_biome', 'Aw_koppen', 'Caatinga_biome'] ... + 15 more


### 3. Model Classes & Helper Functions

In [3]:
# ── Dataset ─────────────────────────────────────────────────────────────────
class TimeSeriesDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        y_t    = torch.tensor(y, dtype=torch.float32)
        self.y = y_t.unsqueeze(1) if y_t.dim() == 1 else y_t
    def __getitem__(self, i): return self.X[i], self.y[i]
    def __len__(self):        return len(self.X)


# ── MLP with Dropout ─────────────────────────────────────────────────────────
class MLPDropout(nn.Module):
    """
    MISO MLP com Dropout em cada camada oculta.
    Com dropout ativo durante inferência → Monte Carlo Dropout.
    """
    _acts = {"relu": nn.ReLU, "tanh": nn.Tanh}

    def __init__(self, n_input, hidden_layers, n_output=1,
                 activation="relu", dropout_p=0.2):
        super().__init__()
        act_cls = self._acts[activation]
        layers, in_dim = [], n_input
        for h in hidden_layers:
            layers += [nn.Linear(in_dim, h), act_cls(), nn.Dropout(p=dropout_p)]
            in_dim = h
        layers.append(nn.Linear(in_dim, n_output))
        self.net = nn.Sequential(*layers)

    def forward(self, x): return self.net(x)


# ── Window builder (MISO) ────────────────────────────────────────────────────
def make_windows(df_scaled, feature_cols, target_col, window_size):
    """Sliding window: X = (N, W*F), y = (N,) next target value."""
    data   = df_scaled[feature_cols].values
    target = df_scaled[target_col].values
    X, y   = [], []
    for i in range(len(data) - window_size):
        X.append(data[i : i + window_size].ravel())
        y.append(target[i + window_size])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


# ── Training with early stopping ─────────────────────────────────────────────
def train_model(model, X_tr, y_tr, X_val, y_val,
                epochs=300, lr=1e-3, patience=PATIENCE):
    """Returns best model (lowest val MSE) and training history."""
    loader   = DataLoader(TimeSeriesDataset(X_tr, y_tr),
                          batch_size=BATCH_SIZE, shuffle=False)
    loss_fn  = nn.MSELoss()
    opt      = torch.optim.Adam(model.parameters(), lr=lr)
    best_val, best_state, wait = np.inf, None, 0
    X_val_t  = torch.tensor(X_val, dtype=torch.float32)
    history  = []

    for epoch in range(1, epochs + 1):
        model.train()
        ep_loss = 0.0
        for Xb, yb in loader:
            opt.zero_grad()
            loss = loss_fn(model(Xb), yb)
            loss.backward()
            opt.step()
            ep_loss += loss.item()
        history.append(ep_loss / len(loader))

        # Validation (model.eval() — dropout off for monitoring)
        model.eval()
        with torch.no_grad():
            val_pred = model(X_val_t).squeeze().numpy()
        val_mse = mean_squared_error(y_val, val_pred)

        if val_mse < best_val - 1e-7:
            best_val   = val_mse
            best_state = deepcopy(model.state_dict())
            wait       = 0
        else:
            wait += 1
            if wait >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history, epoch   # epoch = actual epochs run


# ── Monte Carlo Dropout recursive forecast ───────────────────────────────────
def mc_forecast(model, seed_window, h_total, window_size,
                n_features, n_mc=N_MC):
    """
    seed_window : np.ndarray (window_size, n_features)  — last training window
    Returns     : np.ndarray (h_total, n_mc)  — sample trajectories
    """
    model.train()   # keeps Dropout ACTIVE → stochasticity per forward pass
    all_trajs = np.zeros((h_total, n_mc), dtype=np.float32)

    with torch.no_grad():
        for m in range(n_mc):
            window = seed_window.copy()
            for t in range(h_total):
                x_t   = torch.tensor(window.ravel(), dtype=torch.float32).unsqueeze(0)
                y_hat = model(x_t).item()
                all_trajs[t, m] = y_hat
                new_row         = window[-1].copy()
                new_row[target_idx] = y_hat       # update casos (pos 0)
                window          = np.vstack([window[1:], new_row])

    return all_trajs   # (h_total, n_mc)


### 4. Metrics (WIS, MAE, RMSE, MAPE)

In [4]:
def wis(y_true, median, lower, upper, levels):
    K = len(levels)
    s = 0.5 * np.abs(y_true - median)
    for a in levels:
        IS = (upper[a] - lower[a]
              + (2/a) * np.maximum(0, lower[a] - y_true)
              + (2/a) * np.maximum(0, y_true - upper[a]))
        s += (a/2) * IS
    return float(np.mean(s) / (K + 0.5))


def compute_metrics(y_true, y_pred_med, lower, upper, label):
    yt = np.array(y_true, dtype=float)
    yp = np.array(y_pred_med, dtype=float)
    mask = ~(np.isnan(yt) | np.isnan(yp))
    yt, yp = yt[mask], yp[mask]
    mae  = float(np.mean(np.abs(yt - yp)))
    rmse = float(np.sqrt(np.mean((yt - yp)**2)))
    mape = float(np.nanmean(np.abs((yt - yp) / np.where(yt==0, np.nan, yt))) * 100)
    wis_val = wis(yt, yp,
                  {a: np.array(lower[a])[mask]  for a in PI_LEVELS},
                  {a: np.array(upper[a])[mask]  for a in PI_LEVELS},
                  PI_LEVELS)
    return {"model": label, "MAE": mae, "RMSE": rmse, "MAPE": mape, "WIS": wis_val}


# ── Stationarity helpers (same as classic models) ────────────────────────────
def breusch_pagan_pval(y):
    x = np.hstack([np.ones((len(y),1)), np.arange(len(y)).reshape(-1,1)])
    res = OLS(y, x).fit()
    _, p, _, _ = het_breuschpagan(res.resid, x)
    return p

def n_diffs_kpss(y, max_d=2):
    for d in range(max_d+1):
        s = np.diff(y, n=d) if d > 0 else y
        try:
            _, p, _, _ = kpss(s, regression="c", nlags="auto")
            if p >= 0.05: return d
        except: return d
    return max_d


# ── Plot helpers ─────────────────────────────────────────────────────────────
def save_forecast_plot(train_dates, train_vals, target_dates, target_vals,
                       fc_dates, fc_med, fc_lo, fc_hi, title, path):
    fig, ax = plt.subplots(figsize=(14,4))
    ax.plot(train_dates,  train_vals,  color="#2166ac", lw=0.7, label="Treino")
    ax.plot(target_dates, target_vals, color="black",   lw=1.2, label="Observado")
    pal = ["#f4a582","#d6604d","#b2182b","#67001f"]
    for i, a in enumerate(sorted(fc_lo.keys(), reverse=True)):
        ax.fill_between(fc_dates, fc_lo[a], fc_hi[a],
                        alpha=0.25, color=pal[i], label=f"IP {int(a*100)}%")
    ax.plot(fc_dates, fc_med, color="#d6604d", lw=1.2, ls="--", label="Mediana")
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.tick_params(axis="x", rotation=45, labelsize=7)
    ax.set_ylabel("Casos"); ax.set_title(title, fontsize=9, fontweight="bold")
    ax.legend(fontsize=7, ncol=4); ax.spines[["top","right"]].set_visible(False)
    plt.tight_layout()
    fig.savefig(path, dpi=120, bbox_inches="tight"); plt.close(fig)


### 5. Main Pipeline — Loop por UF e Validation Test

In [5]:
all_forecasts, all_metrics = [], []
txt_lines = []
t0_total  = time.time()

for i_uf, uf_label in enumerate(ufs):

    rd      = hf[hf["uf"] == uf_label].sort_values("date").reset_index(drop=True)
    uf_code = int(uf_code_map.loc[uf_label, "uf_code"])

    txt_lines.append(f"\n{'='*70}")
    txt_lines.append(f"UF: {uf_label}  (uf_code={uf_code})")
    txt_lines.append(f"{'='*70}")
    print(f"[{i_uf+1}/{len(ufs)}] UF: {uf_label}")

    # ── Preliminary analysis on full series ──────────────────────────────────
    y_full  = rd[TARGET].values.astype(float)
    bp_pval = breusch_pagan_pval(np.clip(y_full, 1e-3, None))
    use_log = bp_pval < 0.05
    y_tr_full = np.log1p(y_full) if use_log else y_full
    d_ord   = n_diffs_kpss(y_tr_full)
    inv     = (lambda x: np.expm1(x)) if use_log else (lambda x: np.array(x))

    txt_lines.append(f"Breusch-Pagan p={bp_pval:.4f} → {'log1p' if use_log else 'nível'}")
    txt_lines.append(f"KPSS d sugerido = {d_ord}")

    uf_out_dir = OUT_DIR / uf_label
    uf_out_dir.mkdir(exist_ok=True)

    # ── Historical weekly mean for exog forecast fill ─────────────────────────
    exog_hist = (rd.assign(w=rd["date"].dt.isocalendar().week.astype(int))
                   .groupby("w")[exog_cols].mean())

    # ── Validation loop ───────────────────────────────────────────────────────
    for val_num, train_flag, target_flag in VALIDATIONS:

        txt_lines.append(f"\n  -- Validation {val_num} --")
        train_mask  = rd[train_flag].astype(bool)
        target_mask = rd[target_flag].astype(bool)

        if train_mask.sum() < FREQ or target_mask.sum() == 0:
            txt_lines.append("  [SKIP]"); continue

        tr  = rd[train_mask].sort_values("date")
        tgt = rd[target_mask].sort_values("date")

        y_tr_raw  = tr[TARGET].values.astype(float)
        y_tgt_raw = tgt[TARGET].values.astype(float)

        last_train  = tr["date"].max()
        first_tgt   = tgt["date"].min()
        gap         = int((first_tgt - last_train).days / 7)
        h_total     = gap + len(tgt)

        # ── V4: estende horizonte até EW40/2026 (2026-10-04) ─────────────────
        # O target_4 na base pode estar incompleto pois os dados ainda não foram
        # totalmente reportados. Este bloco garante que a previsão vai até
        # 2026-10-04 (EW40/2026), preenchendo y_tgt_raw com NaN para as semanas
        # futuras ainda sem dados (metrics já ignoram NaN via máscara interna).
        TARGET_V4_END = pd.Timestamp("2026-10-04")
        if val_num == 4:
            required_h = int((TARGET_V4_END - last_train).days / 7)
            if required_h > h_total:
                h_total = required_h
        tgt_dates_ext = pd.date_range(first_tgt, periods=h_total - gap, freq="W-SUN")
        tgt_date_map  = dict(zip(pd.to_datetime(tgt["date"].values), y_tgt_raw))
        y_tgt_raw     = np.array([tgt_date_map.get(d, np.nan) for d in tgt_dates_ext])

        txt_lines.append(f"  Treino : {tr['date'].min().date()} → {last_train.date()} ({len(tr)} sem)")
        txt_lines.append(f"  Gap    : {gap} semanas")
        txt_lines.append(f"  Target : {first_tgt.date()} → {tgt['date'].max().date()} ({len(tgt)} sem)")

        fc_dates = pd.date_range(last_train + pd.Timedelta(weeks=1),
                                 periods=h_total, freq="W-SUN")

        # ── Build future exogenous matrix for forecast ────────────────────────
        fc_df_idx = rd[rd["date"].isin(fc_dates)].set_index("date")
        fc_exog_rows = []
        for d in fc_dates:
            if d in fc_df_idx.index:
                fc_exog_rows.append(fc_df_idx.loc[d, exog_cols].values.astype(float))
            else:
                wk = min(int(d.isocalendar().week), 52)
                fc_exog_rows.append(exog_hist.loc[wk].values
                                    if wk in exog_hist.index else exog_hist.mean().values)
        fc_exog = np.array(fc_exog_rows, dtype=float)  # (h_total, n_exog)

        # ── Transformations on training data ──────────────────────────────────
        # 1) Log transform
        tr_feat = tr[FEATURE_COLS].copy().astype(float)
        if use_log:
            tr_feat[TARGET] = np.log1p(tr_feat[TARGET].values)

        # 2) Differencing (target only, in log space)
        last_log_val = tr_feat[TARGET].values[-1]   # needed to undiff later
        if d_ord == 1:
            tr_feat[TARGET] = tr_feat[TARGET].diff().fillna(0)

        # 3) Fit scaler on training features
        scaler = MinMaxScaler()
        tr_scaled_vals = scaler.fit_transform(tr_feat.values)
        df_tr_scaled   = pd.DataFrame(tr_scaled_vals, columns=FEATURE_COLS)

        # Last scaled value of target (for undiffing forecast)
        last_scaled_target = df_tr_scaled[TARGET].values[-1]

        # 4) Build windows
        n_int_val = max(int(len(df_tr_scaled) * 0.15), 2 * 26)  # 15% as internal val
        n_tr_int  = len(df_tr_scaled) - n_int_val

        # ── Grid search for hyperparameters ──────────────────────────────────
        print(f"    V{val_num} Grid Search...", end=" ", flush=True)
        t0 = time.time()
        best_hp, best_val_mse = None, np.inf

        hp_combinations = list(itertools.product(
            HP_GRID["window_size"], HP_GRID["hidden_layers"],
            HP_GRID["lr"],         HP_GRID["dropout_p"]
        ))

        for (ws, hl, lr, dp) in hp_combinations:
            if n_tr_int <= ws + 5:
                continue
            X_all, y_all = make_windows(df_tr_scaled, FEATURE_COLS, TARGET, ws)
            split        = n_tr_int - ws
            if split <= 0 or len(X_all) - split <= 0:
                continue
            X_tr_s, X_val_s = X_all[:split], X_all[split:]
            y_tr_s, y_val_s = y_all[:split], y_all[split:]

            m_cand = MLPDropout(n_input=ws*n_features, hidden_layers=hl,
                                dropout_p=dp, activation="relu")
            m_cand, _, _ = train_model(m_cand, X_tr_s, y_tr_s, X_val_s, y_val_s,
                                       epochs=EPOCHS_SEARCH, lr=lr)
            m_cand.eval()
            with torch.no_grad():
                val_pred = m_cand(torch.tensor(X_val_s, dtype=torch.float32)).squeeze().numpy()
            val_mse = mean_squared_error(y_val_s, val_pred)
            if val_mse < best_val_mse:
                best_val_mse = val_mse
                best_hp = {"window_size": ws, "hidden_layers": hl, "lr": lr, "dropout_p": dp}

        if best_hp is None:
            best_hp = {"window_size": 8, "hidden_layers": [64,32], "lr":1e-3, "dropout_p":0.2}
        print(f"done ({time.time()-t0:.0f}s) → {best_hp}", flush=True)

        # ── Final training with best hyperparameters ──────────────────────────
        ws_best = best_hp["window_size"]
        X_all, y_all = make_windows(df_tr_scaled, FEATURE_COLS, TARGET, ws_best)
        # Use ALL training data (no internal split) for final model
        split_final = n_tr_int - ws_best
        X_tr_f, X_val_f = X_all[:split_final], X_all[split_final:]
        y_tr_f, y_val_f = y_all[:split_final], y_all[split_final:]

        model_final = MLPDropout(
            n_input    = ws_best * n_features,
            hidden_layers = best_hp["hidden_layers"],
            dropout_p  = best_hp["dropout_p"],
            activation = "relu",
        )
        print(f"    V{val_num} Final training...", end=" ", flush=True)
        t0 = time.time()
        model_final, history, epochs_run = train_model(
            model_final, X_tr_f, y_tr_f,    # ← y_tr_f (labels de treino), não y_val_f
            X_val_f, y_val_f,
            epochs=EPOCHS_FINAL, lr=best_hp["lr"],
        )
        # Retrain on full data (no split) with fixed epochs_run
        X_full, y_full_w = make_windows(df_tr_scaled, FEATURE_COLS, TARGET, ws_best)
        loader_full = DataLoader(TimeSeriesDataset(X_full, y_full_w),
                                 batch_size=BATCH_SIZE, shuffle=False)
        model_final.train()
        opt_full = torch.optim.Adam(model_final.parameters(), lr=best_hp["lr"])
        loss_fn  = nn.MSELoss()
        for _ in range(epochs_run):
            for Xb, yb in loader_full:
                opt_full.zero_grad()
                loss_fn(model_final(Xb), yb).backward()
                opt_full.step()
        print(f"done ({time.time()-t0:.0f}s), {epochs_run} epochs", flush=True)

        # ── Build forecast seed window ────────────────────────────────────────
        seed_window = df_tr_scaled[FEATURE_COLS].values[-ws_best:]  # (ws, F)

        # ── Build future feature matrix in scaled space ───────────────────────
        # For each forecast step, combine predicted casos (scaled+diffed) with
        # exog features (scaled using scaler)
        # We scale the future exog using the fitted scaler:
        # scaler was fitted on [TARGET] + exog_cols columns together
        # We need to pass dummy target values to inverse_transform later
        # → store last window and update target column recursively

        # ── Monte Carlo Dropout forecast ──────────────────────────────────────
        print(f"    V{val_num} MC Dropout ({N_MC} passes)...", end=" ", flush=True)
        t0 = time.time()
        trajs_scaled = mc_forecast(model_final, seed_window, h_total,
                                   ws_best, n_features, N_MC)
        # trajs_scaled: (h_total, N_MC) — in scaled+diffed space

        # ── Inverse transform trajectories ───────────────────────────────────
        def inv_transform_traj(traj_col):
            """traj_col: (h_total,) in scaled+diffed space → original space"""
            # 1) Build full feature matrix for inverse_transform
            #    (target column + dummy zeros for exog)
            tmp = np.zeros((h_total, n_features), dtype=float)
            tmp[:, target_idx] = traj_col
            unscaled = scaler.inverse_transform(tmp)[:, target_idx]
            # 2) Undiff if needed
            if d_ord == 1:
                # last_log_val is the last value before differencing (in log space)
                unscaled = np.concatenate([[last_log_val], unscaled])
                unscaled = np.cumsum(np.diff(unscaled))   # equivalent to cumsum from last_log_val
                # Actually: undiff = cumsum from last_log_val
                unscaled = last_log_val + np.cumsum(unscaled[:h_total] - 0)
                # Simpler:
                running = last_log_val
                result  = []
                for dv in traj_col:
                    # unscale dv first
                    tmp2 = np.zeros((1, n_features))
                    tmp2[0, target_idx] = dv
                    dv_unscaled = scaler.inverse_transform(tmp2)[0, target_idx]
                    running += dv_unscaled
                    result.append(running)
                unscaled = np.array(result)
            # 3) Inverse log
            return inv(unscaled)

        # Correct undiff logic (clean version)
        def inv_traj(traj_scaled_col):
            """(h_total,) scaled → original casos"""
            # Unscale target column only
            tmp = np.zeros((h_total, n_features))
            tmp[:, target_idx] = traj_scaled_col
            unscaled_diff = scaler.inverse_transform(tmp)[:, target_idx]
            # Undiff
            if d_ord == 1:
                vals = [last_log_val + unscaled_diff[0]]
                for i in range(1, h_total):
                    vals.append(vals[-1] + unscaled_diff[i])
                unscaled_log = np.array(vals)
            else:
                unscaled_log = unscaled_diff
            return inv(unscaled_log)

        trajs_orig = np.apply_along_axis(inv_traj, 0, trajs_scaled)
        # trajs_orig: (h_total, N_MC) in original casos space
        trajs_orig = np.clip(trajs_orig, 0, None)   # casos cannot be negative

        fc_median = np.median(trajs_orig, axis=1)
        fc_lower  = {a: np.quantile(trajs_orig, (1-a)/2,    axis=1) for a in PI_LEVELS}
        fc_upper  = {a: np.quantile(trajs_orig, 1-(1-a)/2,  axis=1) for a in PI_LEVELS}

        print(f"done ({time.time()-t0:.0f}s)", flush=True)

        # ── Metrics on target period only ─────────────────────────────────────
        fc_med_tgt = fc_median[gap:]
        fc_lo_tgt  = {a: fc_lower[a][gap:]  for a in PI_LEVELS}
        fc_hi_tgt  = {a: fc_upper[a][gap:]  for a in PI_LEVELS}

        metrics = compute_metrics(y_tgt_raw, fc_med_tgt, fc_lo_tgt, fc_hi_tgt, "MLP")
        print(f"    MAE={metrics['MAE']:.1f}  RMSE={metrics['RMSE']:.1f}  "
              f"MAPE={metrics['MAPE']:.1f}%  WIS={metrics['WIS']:.2f}")

        # ── Save hyperparameter report ────────────────────────────────────────
        arch_str = " → ".join([f"Dense({h}, relu, drop={best_hp['dropout_p']})"
                               for h in best_hp["hidden_layers"]])
        txt_lines.append(f"  MLP Hiperparâmetros:")
        txt_lines.append(f"    Arquitetura     : Input({ws_best*n_features}) → {arch_str} → Output(1)")
        txt_lines.append(f"    window_size     : {ws_best} semanas")
        txt_lines.append(f"    hidden_layers   : {best_hp['hidden_layers']}")
        txt_lines.append(f"    learning_rate   : {best_hp['lr']}")
        txt_lines.append(f"    dropout_p       : {best_hp['dropout_p']}")
        txt_lines.append(f"    epochs_run      : {epochs_run}")
        txt_lines.append(f"    batch_size      : {BATCH_SIZE}")
        txt_lines.append(f"    activation      : relu")
        txt_lines.append(f"    MC passes       : {N_MC}")
        txt_lines.append(f"    use_log         : {use_log}")
        txt_lines.append(f"    d_diff          : {d_ord}")
        txt_lines.append(f"    val_MSE (search): {best_val_mse:.6f}")
        txt_lines.append(f"  Métricas:")
        txt_lines.append(f"    MAE={metrics['MAE']:.1f}  RMSE={metrics['RMSE']:.1f}  "
                         f"MAPE={metrics['MAPE']:.1f}%  WIS={metrics['WIS']:.2f}")

        # ── Save forecast plot ────────────────────────────────────────────────
        save_forecast_plot(
            tr["date"].values, y_tr_raw,
            tgt_dates_ext, y_tgt_raw,
            fc_dates, fc_median, fc_lower, fc_upper,
            f"MLP MISO V{val_num} | {uf_label}  "
            f"(ws={ws_best}, layers={best_hp['hidden_layers']}, WIS={metrics['WIS']:.2f})",
            uf_out_dir / f"v{val_num}_mlp_forecast.png",
        )

        # ── Training curve ────────────────────────────────────────────────────
        fig_h, ax_h = plt.subplots(figsize=(10,3))
        ax_h.plot(history, color="#2166ac", lw=0.9)
        ax_h.set_xlabel("Época"); ax_h.set_ylabel("Train MSE (scaled)")
        ax_h.set_title(f"Learning Curve — V{val_num} | {uf_label}", fontsize=9,
                       fontweight="bold")
        ax_h.spines[["top","right"]].set_visible(False)
        plt.tight_layout()
        fig_h.savefig(uf_out_dir / f"v{val_num}_mlp_learning_curve.png",
                      dpi=120, bbox_inches="tight"); plt.close(fig_h)

        # ── Collect results ───────────────────────────────────────────────────
        all_metrics.append({
            "uf": uf_label, "uf_code": uf_code,
            "validation": val_num, "model": "MLP",
            "window_size": ws_best,
            "hidden_layers": str(best_hp["hidden_layers"]),
            "lr": best_hp["lr"], "dropout_p": best_hp["dropout_p"],
            "epochs_run": epochs_run, "use_log": use_log, "d_diff": d_ord,
            **metrics,
        })
        for i, (d, obs) in enumerate(zip(tgt_dates_ext, y_tgt_raw)):
            row = {"uf": uf_label, "uf_code": uf_code,
                   "validation": val_num, "model": "MLP",
                   "date": d, "y_true": obs, "median": fc_med_tgt[i]}
            for a in PI_LEVELS:
                row[f"lower_{int(a*100)}"] = fc_lo_tgt[a][i]
                row[f"upper_{int(a*100)}"] = fc_hi_tgt[a][i]
            all_forecasts.append(row)

print(f"\nPipeline MLP: {(time.time()-t0_total)/60:.1f} min total")


[1/26] UF: AC
    V1 Grid Search... done (152s) → {'window_size': 8, 'hidden_layers': [64, 32], 'lr': 0.001, 'dropout_p': 0.2}
    V1 Final training... done (6s), 67 epochs
    V1 MC Dropout (500 passes)... done (14s)
    MAE=243.3  RMSE=265.8  MAPE=654.4%  WIS=421.25
    V2 Grid Search... done (72s) → {'window_size': 8, 'hidden_layers': [128, 64, 32], 'lr': 0.001, 'dropout_p': 0.2}
    V2 Final training... done (5s), 40 epochs
    V2 MC Dropout (500 passes)... done (11s)
    MAE=177.1  RMSE=298.9  MAPE=81.6%  WIS=175.73
    V3 Grid Search... done (229s) → {'window_size': 8, 'hidden_layers': [64, 32], 'lr': 0.0005, 'dropout_p': 0.1}
    V3 Final training... done (28s), 263 epochs
    V3 MC Dropout (500 passes)... done (7s)
    MAE=191.1  RMSE=272.5  MAPE=74.7%  WIS=190.18
    V4 Grid Search... done (221s) → {'window_size': 8, 'hidden_layers': [64, 32], 'lr': 0.0005, 'dropout_p': 0.2}
    V4 Final training... done (16s), 130 epochs
    V4 MC Dropout (500 passes)... done (9s)
    MAE=63.

### 6. Export Results

In [6]:
print("Exportando parquets...")
df_metrics   = pd.DataFrame(all_metrics)
df_forecasts = pd.DataFrame(all_forecasts)
df_metrics.to_parquet(OUT_DIR / "metrics_mlp.parquet",    index=False)
df_forecasts.to_parquet(OUT_DIR / "forecasts_mlp.parquet", index=False)

with open(OUT_DIR / "relatorio_mlp.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(txt_lines))

print(f"  metrics_mlp.parquet    → {len(df_metrics):,} linhas")
print(f"  forecasts_mlp.parquet  → {len(df_forecasts):,} linhas")

# Summary table
print("\nResumo médio por Validation Test:")
print(df_metrics.groupby("validation")[["MAE","RMSE","MAPE","WIS"]].mean().round(2).to_string())


Exportando parquets...
  metrics_mlp.parquet    → 104 linhas
  forecasts_mlp.parquet  → 5,408 linhas

Resumo médio por Validation Test:
                    MAE          RMSE         MAPE           WIS
validation                                                      
1           56142468.24  1.658121e+08  14511008.88  1.252505e+08
2               4932.62  8.092300e+03       163.49  4.926220e+03
3               1155.33  1.708390e+03        83.15  1.149630e+03
4                618.72  7.770000e+02      4757.17  6.811400e+02


### 7. Forecast Mosaic by UF

In [7]:
uf_dir = OUT_DIR / "plots_uf"
uf_dir.mkdir(exist_ok=True)
ufs_sorted = sorted(df_forecasts["uf"].unique())
N = len(ufs_sorted); N_COLS = 6; N_ROWS = int(np.ceil(N/N_COLS))
pal = ["#f4a582","#d6604d","#b2182b","#67001f"]

for val_num in [1,2,3,4]:
    df_v = df_forecasts[df_forecasts["validation"]==val_num]
    if df_v.empty: continue
    fig, axes = plt.subplots(N_ROWS, N_COLS, figsize=(24, N_ROWS*3.5),
                              constrained_layout=True)
    axes_flat = axes.flatten()
    for i, uf in enumerate(ufs_sorted):
        ax  = axes_flat[i]
        sub = df_v[df_v["uf"]==uf].sort_values("date")
        if sub.empty: ax.set_visible(False); continue
        ax.plot(sub["date"], sub["y_true"], color="black", lw=1.0, label="Observado")
        for j, a in enumerate(sorted(PI_LEVELS, reverse=True)):
            ax.fill_between(sub["date"], sub[f"lower_{int(a*100)}"],
                            sub[f"upper_{int(a*100)}"], alpha=0.2, color=pal[j])
        ax.plot(sub["date"], sub["median"], color="#d6604d", lw=1.0, ls="--")
        ax.set_title(uf, fontsize=8, fontweight="bold")
        ax.xaxis.set_major_locator(mdates.YearLocator(2))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        ax.tick_params(axis="x", labelsize=6, rotation=45)
        ax.tick_params(axis="y", labelsize=6)
        ax.spines[["top","right"]].set_visible(False)
    for j in range(N, len(axes_flat)): axes_flat[j].set_visible(False)
    fig.suptitle(f"Validation {val_num} — MLP MISO | Casos por UF",
                 fontsize=11, fontweight="bold")
    fig.savefig(uf_dir / f"v{val_num}_mlp_uf_mosaic.png", dpi=130, bbox_inches="tight")
    plt.close(fig)
    print(f"  Salvo: v{val_num}_mlp_uf_mosaic.png")


  Salvo: v1_mlp_uf_mosaic.png
  Salvo: v2_mlp_uf_mosaic.png
  Salvo: v3_mlp_uf_mosaic.png
  Salvo: v4_mlp_uf_mosaic.png


### 8. WIS Heatmap — MLP × UF × Validation Test

In [9]:
for val_num in [1,2,3,4]:
    sub   = df_metrics[df_metrics["validation"]==val_num][["uf","WIS"]].copy()
    if sub.empty: continue
    # Single model row
    pivot = sub.set_index("uf")["WIS"].to_frame().T
    pivot.index = ["MLP"]
    states = pivot.columns.tolist()

    fig, ax = plt.subplots(figsize=(max(12, len(states)*0.75), 2.5))
    im = ax.imshow(pivot.values, cmap="Reds",
                   vmin=np.nanpercentile(pivot.values,5),
                   vmax=np.nanpercentile(pivot.values,95), aspect="auto")
    plt.colorbar(im, ax=ax, fraction=0.02, pad=0.01, label="WIS")
    ax.set_xticks(range(len(states))); ax.set_yticks([0])
    ax.set_xticklabels(states, fontsize=8); ax.set_yticklabels(["MLP"], fontsize=9)
    ax.set_xlabel("State", fontsize=9); ax.set_ylabel("Model", fontsize=9)
    vmax_ann = np.nanpercentile(pivot.values, 80)
    for j, uf in enumerate(states):
        v = pivot.loc["MLP", uf]
        if np.isnan(v): continue
        ax.text(j, 0, f"{v:.1f}", ha="center", va="center",
                fontsize=7, color="white" if v > vmax_ann else "black")
    ax.set_title(f"Validation test - {val_num} (WIS — MLP)", fontsize=11, fontweight="bold")
    plt.tight_layout()
    fig.savefig(OUT_DIR / f"heatmap_wis_mlp_v{val_num}.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Salvo: heatmap_wis_mlp_v{val_num}.png")

print("\n✓ Concluído.")


  Salvo: heatmap_wis_mlp_v1.png
  Salvo: heatmap_wis_mlp_v2.png
  Salvo: heatmap_wis_mlp_v3.png
  Salvo: heatmap_wis_mlp_v4.png

✓ Concluído.
